# STL: Seasonal-Trend Decomposition using LOESS

Este notebook apresenta o metodo **STL** (Cleveland et al., 1990) para decomposicao de series temporais usando a biblioteca `chronobox`.

O STL e um metodo robusto e flexivel que supera muitas limitacoes da decomposicao classica:
- Sazonalidade pode **variar** ao longo do tempo
- Robusto a **outliers** (modo robusto)
- Controle fino via parametros `seasonal`, `trend` e `robust`

O modelo e aditivo: $Y_t = T_t + S_t + R_t$

## Setup e Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chronobox import STL
from chronobox.visualization import plot_decomposition

import sys
sys.path.insert(0, '..')
from utils.plot_helpers import plot_stl_diagnostics, plot_seasonal_subseries
from utils.data_generators import generate_additive_seasonal, generate_complex_seasonal

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 6)

## 1. O que e LOESS?

LOESS (LOcally Estimated Scatterplot Smoothing) e um metodo de **regressao local**:

- Para cada ponto $x_0$, ajusta um polinomio ponderado aos vizinhos
- Pesos decrescem com a distancia (kernel tricube)
- O parametro `window` controla quantos vizinhos sao usados

No STL, LOESS e aplicado em dois contextos:
- **Suavizacao sazonal** (s.window / `seasonal`): controla quao rapido a sazonalidade pode variar
- **Suavizacao de tendencia** (t.window / `trend`): controla a suavidade da tendencia

## 2. Algoritmo STL - Visao Geral

O STL usa dois loops aninhados:

**Inner loop** (repete `n_inner` vezes, default 2):
1. **Detrending**: remove tendencia atual → $Y_t - T_t$
2. **Cycle-subseries smoothing**: aplica LOESS a cada subserie sazonal (todos os janeiros, todos os fevereiros, etc.)
3. **Low-pass filter**: media movel seguida de LOESS para extrair tendencia do componente sazonal
4. **Deseasonalizing**: subtrai sazonal → $Y_t - S_t$
5. **Trend smoothing**: aplica LOESS a serie dessazonalizada

**Outer loop** (repete `n_outer` vezes, 0 se nao robusto):
- Calcula **pesos de robustez** baseados nos residuos (bisquare)
- Pontos com residuos grandes recebem peso menor
- O inner loop repete com esses pesos

## 3. Carregando os Dados CO2

Usaremos o dataset de concentracao de CO2 atmosferico em Mauna Loa - uma serie classica com tendencia nao-linear e sazonalidade.

In [ ]:
# Carregar dados CO2
co2 = pd.read_csv('../data/co2.csv', parse_dates=['date'])
y_co2 = co2['co2'].values
dates_co2 = pd.DatetimeIndex(co2['date'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(dates_co2, y_co2, 'k-', linewidth=0.8)
ax.set_title('Concentracao de CO2 - Mauna Loa (ppm)')
ax.set_ylabel('CO2 (ppm)')
ax.set_xlabel('Ano')
plt.show()

print(f"Observacoes: {len(y_co2)}")
print(f"Periodo: {dates_co2[0].strftime('%Y-%m')} a {dates_co2[-1].strftime('%Y-%m')}")

## 4. STL Basico

Aplicando STL com parametros padrao.

In [ ]:
# STL com parametros padrao
stl_default = STL(period=12, seasonal=7)
result_default = stl_default.fit(y_co2)

print(result_default.summary())

In [ ]:
# Visualizacao dos 4 componentes
fig = plot_decomposition(result_default, title='STL Decomposition - CO2 (default)', dates=dates_co2)
plt.show()

### Interpretacao Visual

- **Observed**: serie CO2 com clara tendencia ascendente e ciclo anual
- **Trend**: captura a tendencia nao-linear (aceleracao do aumento de CO2 ao longo das decadas)
- **Seasonal**: ciclo anual dominado por fotossintese do hemisferio norte (CO2 cai no verao, sobe no inverno)
- **Remainder**: variacao residual, idealmente pequena e sem estrutura

Note que o STL nao perde observacoes nas extremidades (diferente da decomposicao classica).

## 5. Parametro `seasonal` (s.window)

O parametro `seasonal` controla a **janela LOESS para a componente sazonal**:
- Valor **menor** (ex: 7): sazonalidade pode variar rapidamente ano a ano
- Valor **maior** (ex: 35): sazonalidade mais estavel, proxima de fixa
- Valor muito grande → equivale a decomposicao classica (sazonalidade constante)

Deve ser **impar** e >= 7.

In [ ]:
# Comparacao com diferentes valores de seasonal
seasonal_values = [7, 13, 25, 51]

fig, axes = plt.subplots(len(seasonal_values), 1, figsize=(14, 12), sharex=True)
fig.suptitle('Efeito do parametro seasonal (s.window) na componente sazonal', fontsize=14)

for ax, s_win in zip(axes, seasonal_values):
    stl_s = STL(period=12, seasonal=s_win)
    res_s = stl_s.fit(y_co2)
    ax.plot(dates_co2, res_s.seasonal, 'g-', linewidth=0.8)
    ax.set_ylabel(f's={s_win}')
    ax.set_title(f'seasonal={s_win}', fontsize=10, loc='left')

fig.tight_layout()
plt.show()

### Interpretacao

- Com `seasonal=7`: a amplitude sazonal varia visivelmente ao longo dos anos (mais flexivel)
- Com `seasonal=51`: a sazonalidade e quase constante, como na decomposicao classica
- O CO2 tem amplitude sazonal levemente crescente, entao valores intermediarios (13-25) sao adequados

## 6. Parametro `trend` (t.window)

O parametro `trend` controla a **janela LOESS para a tendencia**:
- Valor **menor**: tendencia mais flexivel (pode capturar ciclos de medio prazo)
- Valor **maior**: tendencia mais suave
- Default: calculado automaticamente como $\lceil 1.5 \cdot m / (1 - 1.5/s) \rceil$ (arredondado para impar)

Deve ser **impar** e maior que o periodo.

In [ ]:
# Comparacao com diferentes valores de trend
trend_values = [13, 25, 51, 101]

fig, axes = plt.subplots(len(trend_values), 1, figsize=(14, 12), sharex=True)
fig.suptitle('Efeito do parametro trend (t.window) na componente de tendencia', fontsize=14)

for ax, t_win in zip(axes, trend_values):
    stl_t = STL(period=12, seasonal=7, trend=t_win)
    res_t = stl_t.fit(y_co2)
    ax.plot(dates_co2, res_t.trend, 'b-', linewidth=1.2)
    ax.plot(dates_co2, y_co2, 'k-', linewidth=0.3, alpha=0.5)
    ax.set_ylabel(f't={t_win}')
    ax.set_title(f'trend={t_win}', fontsize=10, loc='left')

fig.tight_layout()
plt.show()

### Interpretacao

- `trend=13`: tendencia muito flexivel, captura flutuacoes de curto prazo
- `trend=101`: tendencia bem suave, quase linear em cada decada
- A escolha depende do objetivo: se quer isolar ciclos de medio prazo, use `trend` menor

## 7. Modo Robusto

O modo `robust=True` torna o STL menos sensivel a **outliers**:
- Calcula pesos baseados no tamanho dos residuos (funcao bisquare)
- Outliers recebem peso zero, nao influenciando a estimacao
- Usa `n_outer=15` iteracoes externas por padrao

In [ ]:
# Criar dados com outliers
y_outlier = y_co2.copy()
np.random.seed(42)
outlier_idx = np.random.choice(len(y_co2), size=10, replace=False)
y_outlier[outlier_idx] += np.random.uniform(5, 15, size=10) * np.random.choice([-1, 1], size=10)

# STL normal vs robusto
stl_normal = STL(period=12, seasonal=7, robust=False)
stl_robust = STL(period=12, seasonal=7, robust=True)

res_normal = stl_normal.fit(y_outlier)
res_robust = stl_robust.fit(y_outlier)

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle('STL Normal (esquerda) vs Robusto (direita) - Dados com Outliers', fontsize=14)

axes[0, 0].plot(dates_co2, y_outlier, 'k-', linewidth=0.5)
axes[0, 0].plot(dates_co2[outlier_idx], y_outlier[outlier_idx], 'ro', markersize=5)
axes[0, 0].set_title('Dados com outliers')
axes[0, 1].plot(dates_co2, y_outlier, 'k-', linewidth=0.5)
axes[0, 1].plot(dates_co2[outlier_idx], y_outlier[outlier_idx], 'ro', markersize=5)
axes[0, 1].set_title('Dados com outliers')

axes[1, 0].plot(dates_co2, res_normal.trend, 'b-', linewidth=1)
axes[1, 0].set_title('Tendencia (Normal)')
axes[1, 1].plot(dates_co2, res_robust.trend, 'b-', linewidth=1)
axes[1, 1].set_title('Tendencia (Robusto)')

axes[2, 0].plot(dates_co2, res_normal.remainder, 'r-', linewidth=0.8)
axes[2, 0].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[2, 0].set_title('Residuo (Normal)')
axes[2, 1].plot(dates_co2, res_robust.remainder, 'r-', linewidth=0.8)
axes[2, 1].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[2, 1].set_title('Residuo (Robusto)')

fig.tight_layout()
plt.show()

In [ ]:
# Pesos de robustez - outliers recebem peso baixo
if res_robust.weights is not None:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(dates_co2, res_robust.weights, 'purple', linewidth=0.8)
    ax.plot(dates_co2[outlier_idx], res_robust.weights[outlier_idx], 'ro', markersize=6,
            label='Outliers')
    ax.set_title('Pesos de Robustez (1 = confiavel, 0 = outlier)')
    ax.set_ylabel('Peso')
    ax.set_ylim(-0.05, 1.05)
    ax.legend()
    plt.show()
    
    print(f"Peso medio nos outliers: {res_robust.weights[outlier_idx].mean():.4f}")
    print(f"Peso medio nos demais:   {np.delete(res_robust.weights, outlier_idx).mean():.4f}")

### Interpretacao (Robustez)

- No modo **normal**, outliers distorcem a tendencia e a sazonalidade localmente
- No modo **robusto**, outliers sao absorvidos pelo residuo (recebem peso ~0)
- Os **pesos** mostram quais observacoes o STL considera confiaveis
- Use `robust=True` quando suspeitar de dados anomalos ou erros de medicao

## 8. Vantagens do STL sobre Decomposicao Classica

| Aspecto | Classica | STL |
|---------|----------|-----|
| Sazonalidade | Fixa | Pode variar |
| Outliers | Sensivel | Robusto (opcional) |
| Extremidades | Perde dados (NaN) | Sem perda |
| Flexibilidade | Nenhuma | Controle via parametros |
| Modelo | Aditivo ou multiplicativo | Aditivo (multiplicativo via log) |

In [ ]:
# Comparacao direta: STL vs Classica no mesmo dataset
from chronobox import ClassicalDecomposition

cd = ClassicalDecomposition(period=12, model='additive')
stl = STL(period=12, seasonal=7)

res_cd = cd.fit(y_co2)
res_stl = stl.fit(y_co2)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Classica (vermelho) vs STL (azul) - CO2', fontsize=14)

axes[0].plot(dates_co2, res_cd.trend, 'r-', label='Classica', linewidth=1)
axes[0].plot(dates_co2, res_stl.trend, 'b-', label='STL', linewidth=1)
axes[0].set_title('Tendencia')
axes[0].legend()

axes[1].plot(dates_co2, res_cd.seasonal, 'r-', label='Classica', linewidth=0.8)
axes[1].plot(dates_co2, res_stl.seasonal, 'b-', label='STL', linewidth=0.8)
axes[1].set_title('Sazonal')
axes[1].legend()

axes[2].plot(dates_co2, res_cd.remainder, 'r-', label='Classica', linewidth=0.8)
axes[2].plot(dates_co2, res_stl.remainder, 'b-', label='STL', linewidth=0.8)
axes[2].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[2].set_title('Residuo')
axes[2].legend()

fig.tight_layout()
plt.show()

### Interpretacao da Comparacao

- **Tendencia**: STL captura a aceleracao nao-linear com mais fidelidade; classica perde as extremidades
- **Sazonal**: na classica, a sazonalidade e identica todos os anos; o STL permite variacao gradual
- **Residuo**: o STL geralmente produz residuos menores por ser mais flexivel

## 9. STL com Diferentes Configuracoes - Resumo

Criando uma tabela comparativa de metricas para diferentes configuracoes.

In [ ]:
# Tabela comparativa de configuracoes
configs = [
    {'seasonal': 7, 'trend': None, 'robust': False, 'label': 'Default (s=7)'},
    {'seasonal': 13, 'trend': None, 'robust': False, 'label': 's=13'},
    {'seasonal': 25, 'trend': None, 'robust': False, 'label': 's=25'},
    {'seasonal': 7, 'trend': 25, 'robust': False, 'label': 's=7, t=25'},
    {'seasonal': 7, 'trend': 51, 'robust': False, 'label': 's=7, t=51'},
    {'seasonal': 7, 'trend': None, 'robust': True, 'label': 's=7, robust'},
]

print(f"{'Config':<20} {'Resid Std':>10} {'Resid Max':>10} {'Trend Range':>12} {'Seas Amp':>10}")
print('-' * 65)

for cfg in configs:
    stl_cfg = STL(period=12, seasonal=cfg['seasonal'],
                  trend=cfg['trend'], robust=cfg['robust'])
    res = stl_cfg.fit(y_co2)
    r_std = np.std(res.remainder)
    r_max = np.max(np.abs(res.remainder))
    t_range = np.nanmax(res.trend) - np.nanmin(res.trend)
    s_amp = np.max(res.seasonal) - np.min(res.seasonal)
    print(f"{cfg['label']:<20} {r_std:>10.4f} {r_max:>10.4f} {t_range:>12.4f} {s_amp:>10.4f}")

## Exercicio

### Exercicio 1: Efeito dos Parametros no CO2

1. Aplique STL ao `co2.csv` com `seasonal` = 7, 15 e 35
2. Para cada configuracao, plote os 4 componentes
3. Compare a amplitude sazonal ao longo do tempo
4. Qual valor de `seasonal` captura melhor a mudanca na amplitude sazonal?

### Exercicio 2: Robustez a Outliers

1. Adicione 5 outliers ao `co2.csv` (pontos com +10 ppm)
2. Aplique STL com `robust=False` e `robust=True`
3. Compare as tendencias: qual e mais afetada pelos outliers?
4. Analise os pesos de robustez: os outliers foram detectados?

In [ ]:
# Exercicio 1 - Espaco para resolucao
# Dica: STL(period=12, seasonal=7) vs seasonal=15 vs seasonal=35

# Seu codigo aqui:


In [ ]:
# Exercicio 2 - Espaco para resolucao
# Dica: y_outlier = y_co2.copy(); y_outlier[indices] += 10
# Compare STL(robust=False) vs STL(robust=True)

# Seu codigo aqui:
